# Module 5 — Code Along: Driving the API with `requests` (the bank-accounts story, Day 2)

In **Module 4** we made `BankAccount` a validated Pydantic model: bad data is rejected at construction, CSV strings coerce cleanly, and `AccountCreate`/`AccountUpdate` give us request shapes. We had a *model* — but no way to talk to a server.

**Today we drive accounts over HTTP.** We learn the real `requests` API shape — verbs, status codes, sessions, auth, retries — and wrap it in a small `AccountClient` that returns those same Pydantic models.

Same canonical account shape and the same owners (Ada, Lin, Sam) as before:

```python
id: int, owner: str, account_type: str, balance: float, is_active: bool, tags: list[str]
```

**Key trick — no live server needed.** A real `AccountClient` would hold a `requests.Session()`. Here we *inject a `FakeSession`* instead, so every cell runs headless with zero network. The client never knows the difference — that injection seam is exactly the mocking pattern Day 3 leans on. The lab hits the **real** Day-1 FastAPI server with a real `requests.Session()`.

Run top to bottom. Cells are self-contained and re-runnable.

## Setup — paste, don't narrate

This one cell is **scaffolding, not a lesson.** `FakeResponse` + `FakeSession` let us run the real `requests` code with **no live server** — every later cell injects one of these instead of a `requests.Session()`. The lab hits the real Day-1 server with a real session; here we just need something to talk to.

The fake is **configured, not subclassed**: you hand it `{ "METHOD /path": [outcome, outcome, ...] }`. Each call pops the next outcome for that route — a `FakeResponse` to return, or an exception to raise (for the retry/429 demos). When a queue runs dry it repeats its last item, so steady-state `GET`s keep working. **Paste it, run it, move on** — we'll narrate the cells *below* it.

In [ ]:
from urllib.parse import urlsplit


class FakeResponse:
    """Mimics the bits of requests.Response we use: .status_code, .ok, .json(), .raise_for_status(), .headers."""
    def __init__(self, status_code, json_body, headers=None):
        self.status_code = status_code
        self._json = json_body
        self.headers = headers or {}          # e.g. {"Retry-After": "0.01"} for the 429 demo

    @property
    def ok(self):                             # requests computes .ok as status < 400
        return self.status_code < 400

    def json(self):                           # parsed response body
        return self._json

    def raise_for_status(self):               # the real method raises on 4xx/5xx
        # NOTE: real requests raises requests.HTTPError here; the fake raises a plain
        #       RuntimeError to keep the demo dependency-light.
        if not self.ok:
            raise RuntimeError(f"HTTP {self.status_code}")


class FakeSession:
    """ONE configurable stand-in for requests.Session — no subclasses.

    routes maps "METHOD /path" -> a QUEUE (list) of scripted outcomes consumed in order.
    Each outcome is a FakeResponse to return, OR an Exception instance to raise (retry/429 demos).
    When a queue has one item left it repeats it, so steady-state GETs keep working.
    """
    def __init__(self, routes):
        # normalise every value to a list (a queue); a bare FakeResponse becomes [resp].
        self.routes = {k: (v if isinstance(v, list) else [v]) for k, v in routes.items()}
        self.headers = {}                     # so session.headers.update(...) works
        self.sent = []                        # record calls, to inspect json= etc.

    def request(self, method, url, **kwargs):
        path = urlsplit(url).path                       # derive path from the full URL
        self.sent.append((method, path, kwargs))        # remember the call for assertions
        queue = self.routes[f"{method} {path}"]         # the queue for this route
        item = queue.pop(0) if len(queue) > 1 else queue[0]   # pop, but keep the last as steady state
        if isinstance(item, Exception):
            raise item                                  # scripted transport failure (ConnectionError, ...)
        return item                                     # scripted FakeResponse

    # convenience wrappers, exactly like requests.Session().get/post/...
    def get(self, url, **kw):  return self.request("GET", url, **kw)
    def post(self, url, **kw): return self.request("POST", url, **kw)

## The HTTP verbs you actually use

Every call is one **verb** + one URL, and the verb is a promise about what it does. Two properties of that promise drive everything else: **safe** = read-only (changes nothing); **idempotent** = sending it N times leaves the same end state. Idempotency is *why retries are safe* — a retry re-sends the identical request, fine for `GET`/`PUT`/`DELETE`, risky for a `POST` that creates a new row each time.

In [ ]:
# A tiny lookup table is enough to reason about retry safety per verb.
# Rule: a retry only re-sends the same request safely when the verb is idempotent.
VERBS = [
    ("GET",    True,  True,  "read an account"),
    ("POST",   False, False, "create an account"),
    ("PUT",    False, True,  "replace the whole account"),
    ("PATCH",  False, False, "partial update"),
    ("DELETE", False, True,  "remove an account"),
]

for verb, safe, idem, use in VERBS:
    print(f"{verb:7} safe={safe!s:5} idempotent={idem!s:5}  # {use}")
# GET     safe=True  idempotent=True   # read an account
# POST    safe=False idempotent=False  # create an account
# PUT     safe=False idempotent=True   # replace the whole account
# PATCH   safe=False idempotent=False  # partial update
# DELETE  safe=False idempotent=True   # remove an account

## Status codes — whose fault is it?

**❓ Predict first:** of 404, 422, 503 — which should the client retry?

The first digit tells you who broke and what to do: **2xx** ok, **3xx** redirect, **4xx** YOUR fault (bad request/auth/payload), **5xx** the SERVER's fault. The entire retry policy falls out of this: a 4xx repeats identically so retrying is pointless; a 5xx or a dropped connection is often transient, so a retry can recover.

In [ ]:
def classify(code: int) -> str:
    """Map a status code to an action. This *is* the retry policy in one place."""
    if 200 <= code < 300:
        return "ok"
    if 300 <= code < 400:
        return "redirect"
    if 400 <= code < 500:
        return "client-error (YOUR fault — do NOT retry)"
    return "server-error (retry may recover)"   # 5xx

for code in (200, 201, 304, 404, 422, 503):
    print(code, "->", classify(code))
# 200 -> ok
# 201 -> ok
# 304 -> redirect
# 404 -> client-error (YOUR fault — do NOT retry)
# 422 -> client-error (YOUR fault — do NOT retry)
# 503 -> server-error (retry may recover)

# Rule of thumb we will encode later: retry 5xx + network errors; never retry 4xx.
print("retry?", {c: classify(c).startswith("server") for c in (404, 422, 503)})
# retry? {404: False, 422: False, 503: True}

## `requests` — one good pattern (faked so it runs)

Production code holds a `requests.Session()`: it pools TCP connections and carries default headers across calls. The shape is always four moves — build the session, set headers, send with `json=payload, timeout=5.0`, then `raise_for_status()` + `.json()`. **`timeout` is not optional** — without it a stalled server hangs you forever.

We inject the `FakeSession` from setup instead of `requests.Session()`. It mimics the real API shape exactly — `json=` carries a dict, `raise_for_status()` raises on 4xx/5xx — so everything we learn transfers to the real session unchanged.

In [ ]:
# Exercise the 'one good pattern' against the injected FakeSession.
session = FakeSession({
    "POST /accounts": FakeResponse(201, {"id": 1, "owner": "Ada", "balance": 1500.0}),
})
session.headers.update({"X-Trace-Id": "abc"})      # default header on every call

payload = {"id": 1, "owner": "Ada", "balance": 1500.0}
resp = session.post("http://localhost:8000/accounts", json=payload, timeout=5.0)
resp.raise_for_status()                            # would raise on 4xx/5xx; 201 is fine
print(resp.status_code, resp.json()["owner"])      # 201 Ada

# json= really did carry our dict through (real requests serialises it + sets Content-Type).
print("sent json=", session.sent[0][2]["json"])    # sent json= {'id': 1, 'owner': 'Ada', 'balance': 1500.0}

# raise_for_status() is the safety net: a 4xx response raises instead of sliding through.
bad = FakeResponse(422, {"detail": "balance must be >= 0"})
try:
    bad.raise_for_status()
except RuntimeError as exc:
    print("raise_for_status ->", exc)              # raise_for_status -> HTTP 422

## Auth: tokens & headers

Auth is just a header you set on the session **once** — every later call carries it. The common forms: a `Bearer` token, HTTP Basic (`auth=(user, pass)`), or an API-key header. Never put the key in the **query string** — it leaks into server logs and browser history. And never put the secret in source: read it from `os.environ` and never `print` it.

In [ ]:
import os

# Secrets come from the environment, not the source file. (Set a fake one for the demo.)
os.environ.setdefault("CATALOG_TOKEN", "s3cret-token")
token = os.environ["CATALOG_TOKEN"]

session = FakeSession({"GET /accounts": FakeResponse(200, [])})

# Bearer token — set ONCE on the session; every later request carries it automatically.
session.headers.update({"Authorization": f"Bearer {token}"})
print("Authorization set:", session.headers["Authorization"].startswith("Bearer "))   # True

# Never print the secret itself — mask it. (It would otherwise live in CI logs forever.)
print("token (masked):", token[:2] + "***")   # token (masked): s3***

# API key in a HEADER (good) vs the QUERY STRING (bad — leaks into logs/history).
good = {"X-API-Key": token}                 # header: not logged with the URL
bad_url = f"http://localhost:8000/accounts?api_key={token}"   # DON'T: ends up in logs
print("header keys:", list(good))           # header keys: ['X-API-Key']
print("query leaks secret into URL:", "api_key=" in bad_url)   # True

## Wrap it in `AccountClient`

A CRUD method per verb invites copy-paste drift. Instead, **one private `_request` funnel** owns URL-joining, the timeout default, and the error check; every public method calls it. A non-2xx response becomes a small `APIError(status, detail)` callers can catch. The client takes an **injected `session`** — defaulting to a real `requests.Session()` in production, but we always hand it a `FakeSession` here.

In [ ]:
import requests   # imported for the real default + exception types; transport stays faked


class APIError(Exception):
    """A non-2xx response, surfaced as a catchable error with the status + detail."""
    def __init__(self, status, detail):
        super().__init__(f"{status}: {detail}")
        self.status = status
        self.detail = detail


class AccountClient:
    def __init__(self, base_url, *, timeout=5.0, session=None):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        # Production: a real pooled Session. Tests/notebook: an injected FakeSession.
        self._session = session or requests.Session()

    def _request(self, method, path, **kwargs):
        """The single funnel: every public method goes through here."""
        kwargs.setdefault("timeout", self.timeout)        # never let a call hang forever
        resp = self._session.request(method, f"{self.base_url}{path}", **kwargs)
        if not resp.ok:                                   # 4xx/5xx -> raise APIError
            detail = resp.json().get("detail", "") if hasattr(resp, "json") else ""
            raise APIError(resp.status_code, detail)
        return resp


# Inject a FakeSession with two routes: one good, one that 409s.
session = FakeSession({
    "GET /accounts":  FakeResponse(200, [{"id": 1, "owner": "Ada"}]),
    "POST /accounts": FakeResponse(409, {"detail": "id 1 already exists"}),
})
client = AccountClient("http://localhost:8000", session=session)

# Happy path funnels through _request and returns the response object.
print(client._request("GET", "/accounts").json())   # [{'id': 1, 'owner': 'Ada'}]

# A 409 becomes a clean APIError carrying status + detail — easy to catch upstream.
try:
    client._request("POST", "/accounts", json={"id": 1, "owner": "Ada"})
except APIError as exc:
    print("APIError:", exc.status, "->", exc.detail)   # APIError: 409 -> id 1 already exists

## Typed returns, not dicts

The wire speaks dicts; your code should not. Each method validates the JSON back into the **Pydantic models from Module 4**, so callers stay in `BankAccount`-land — and a malformed server response fails loudly at the boundary, not three functions later. `list_accounts -> list[BankAccount]`; `create_account(payload: AccountCreate) -> BankAccount`.

In [ ]:
from pydantic import BaseModel, Field


# The same shapes we built in Module 4 (kept minimal here).
class AccountBase(BaseModel):
    owner: str
    account_type: str = "savings"
    balance: float = Field(default=0.0, ge=0)
    is_active: bool = True
    tags: list[str] = Field(default_factory=list)

class AccountCreate(AccountBase):
    id: int = Field(ge=1)

class BankAccount(AccountBase):
    id: int


class TypedAccountClient(AccountClient):
    def list_accounts(self) -> list[BankAccount]:
        data = self._request("GET", "/accounts").json()
        return [BankAccount.model_validate(r) for r in data]   # dicts -> models, validated

    def create_account(self, payload: AccountCreate) -> BankAccount:
        data = self._request("POST", "/accounts", json=payload.model_dump()).json()
        return BankAccount.model_validate(data)


# FakeSession returns our three canonical owners as raw dicts (as a server would).
session = FakeSession({
    "GET /accounts": FakeResponse(200, [
        {"id": 1, "owner": "Ada", "balance": 1500.0, "tags": ["primary"]},
        {"id": 2, "owner": "Lin", "account_type": "checking"},
        {"id": 3, "owner": "Sam", "is_active": False},
    ]),
    "POST /accounts": FakeResponse(201, {"id": 4, "owner": "Ada", "balance": 200.0}),
})
client = TypedAccountClient("http://localhost:8000", session=session)

accounts = client.list_accounts()
print(type(accounts[0]).__name__, [a.owner for a in accounts])   # BankAccount ['Ada', 'Lin', 'Sam']
print(accounts[1].account_type, accounts[2].is_active)           # checking False

created = client.create_account(AccountCreate(id=4, owner="Ada", balance=200.0))
print(type(created).__name__, created.id, created.owner)         # BankAccount 4 Ada

## When to retry — and when not

Reuse **Day-1's `@retry`** on the `_request` funnel, scoped to **transient** failures only: connection drops and timeouts. The exception tuple *is* the policy — a 4xx raises an `APIError`, never a `ConnectionError`/`Timeout`, so it can never be retried (400/401/403/409/422 fail fast, correctly). Below, the **same configurable `FakeSession`** is queued to raise a connection error twice, then succeed — the retry recovers on attempt 3.

In [ ]:
import time
from functools import wraps
from requests.exceptions import ConnectionError, Timeout   # real transport types


# The Day-1 M3 @retry, inlined. Retries ONLY the listed exception types.
def retry(times=3, delay=0.0, exceptions=(Exception,)):
    def decorate(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions:
                    if attempt == times:
                        raise            # out of attempts -> let it propagate
                    time.sleep(delay)    # transient -> wait, then try again
        return wrapper
    return decorate


class RetryingClient(AccountClient):
    @retry(times=3, delay=0.0, exceptions=(ConnectionError, Timeout))
    def _request(self, method, path, **kwargs):
        return super()._request(method, path, **kwargs)


# Queue the ONE FakeSession to fail twice then succeed — no FlakySession subclass needed.
flaky = FakeSession({"GET /accounts": [
    ConnectionError("connection refused"),     # attempt 1 -> raise (transient)
    ConnectionError("connection refused"),     # attempt 2 -> raise (transient)
    FakeResponse(200, [{"id": 1, "owner": "Ada"}]),   # attempt 3 -> succeed
]})
client = RetryingClient("http://localhost:8000", session=flaky)
data = client._request("GET", "/accounts").json()
print("recovered after retries ->", data)   # recovered after retries -> [{'id': 1, 'owner': 'Ada'}]

# But a 4xx raises APIError, NOT ConnectionError/Timeout — so @retry leaves it alone.
session = FakeSession({"POST /accounts": FakeResponse(422, {"detail": "balance >= 0"})})
client = RetryingClient("http://localhost:8000", session=session)
try:
    client._request("POST", "/accounts", json={"owner": "Sam", "balance": -1})
except APIError as exc:
    print("not retried (4xx):", exc.status)   # not retried (4xx): 422

## Where this goes next — Module 6

We now have an `AccountClient` that **speaks HTTP and returns Pydantic models**: one `_request` funnel, `APIError` on failures, `@retry` scoped to transient errors only, typed CRUD on top. The `FakeSession` let us learn the real `requests` shape with no server — and that same injection seam is how Day 3 mocks the network.

**Next (Module 6)** we point this client at a **CSV of accounts** and bulk-load them: validate each row through `AccountCreate`, send the good ones, and emit a **report** that keeps validation errors, API errors, and successes in separate buckets — the artifact an operator uses to fix row 18 and re-run.

The lab takes this exact pattern and builds a `catalog/client.py` `APIClient` for `Product` against the **real** Day-1 FastAPI server, with a real `requests.Session()`.